In [ ]:
!pip install -q transformers datasets evaluate accelerate tf-keras

import os
import shutil
import random
import pandas as pd
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset, DatasetDict
from transformers import AutoImageProcessor
from tqdm.notebook import tqdm
from torchvision.transforms import (
    Compose,
    RandomResizedCrop,
    RandomHorizontalFlip,
    RandomApply,
    ColorJitter,
    RandomRotation,
    GaussianBlur,
    ToTensor,
    Normalize,
    Resize,
    CenterCrop,
)
from transformers import AutoModelForImageClassification
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import torch.nn as nn


In [ ]:
df = pd.read_csv("data/articles_image.csv")

df


In [ ]:
def organize_images_by_label_split(df: pd.DataFrame, output_dir: str, train_ratio: float = 0.8, seed: int = 42):
    """
    Organize images into train/test subfolders by label, using a specified train ratio.

    Args:
        df (pd.DataFrame):
            DataFrame containing:
              - "filepath": path to the image file
              - "label": the class/category for that image
        output_dir (str):
            Path to the output directory. Will create "train/" and "test/" subfolders,
            each containing a subfolder per label.
        train_ratio (float):
            Fraction of images per label to go into train. The rest go into test.
            Defaults to 0.8 (i.e., 80% train, 20% test).
        seed (int):
            Random seed for reproducibility.
    """
    random.seed(seed)

    # Create the top-level output directories
    train_dir = os.path.join(output_dir, "train")
    val_dir = os.path.join(output_dir, "val")
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(val_dir, exist_ok=True)

    # Get unique labels
    labels = df["label"].unique()

    for label in labels:
        # Filter rows for this label
        label_df = df[df["label"] == label]

        # Get list of filepaths for this label
        filepaths = label_df["filepath"].tolist()

        # Shuffle them to ensure a random split
        random.shuffle(filepaths)

        # Split index
        split_index = int(len(filepaths) * train_ratio)
        train_files = filepaths[:split_index]
        test_files = filepaths[split_index:]

        # Create subfolders: train/<label>, test/<label>
        label_train_dir = os.path.join(train_dir, str(label))
        label_val_dir = os.path.join(val_dir, str(label))
        os.makedirs(label_train_dir, exist_ok=True)
        os.makedirs(label_val_dir, exist_ok=True)

        # Copy the training images
        for filepath in train_files:
            if os.path.isfile(filepath):
                shutil.copy2(filepath, label_train_dir)
            else:
                print(f"Warning: file not found -> {filepath}")

        # Copy the testing images
        for filepath in test_files:
            if os.path.isfile(filepath):
                shutil.copy2(filepath, label_val_dir)
            else:
                print(f"Warning: file not found -> {filepath}")

        print(f"Label '{label}': {len(train_files)} train, {len(test_files)} test")

    print("Organization into train/test completed!")


organize_images_by_label_split(df, "./images_split/")


In [ ]:
{
    "0": "baby care",
    "1": "beauty, personal care",
    "2": "computers",
    "3": "home decor, festive needs",
    "4": "home furnishing",
    "5": "kitchen, dining",
    "6": "watches",
}


In [ ]:
dataset = load_dataset("imagefolder", data_dir="./images_split")

print(dataset["train"].features)


In [ ]:
example = dataset["train"][0]
example["image"]


In [ ]:
labels = dataset["train"].features["label"].names
print(labels)


In [ ]:
id2label = {k: v for k, v in enumerate(labels)}
label2id = {v: k for k, v in enumerate(labels)}
print(id2label)


In [ ]:
image_processor = AutoImageProcessor.from_pretrained("facebook/convnext-small-224", use_fast=True)


In [ ]:
print(image_processor)


In [ ]:
normalize = Normalize(mean=image_processor.image_mean, std=image_processor.image_std)

train_transform = Compose(
    [
        RandomResizedCrop(image_processor.size["shortest_edge"]),
        RandomHorizontalFlip(p=0.5),
        RandomApply(
            [ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1)],
            p=0.5,
        ),
        RandomRotation(degrees=15),
        RandomApply([GaussianBlur(kernel_size=(3, 3))], p=0.2),
        ToTensor(),
        normalize,
    ]
)
val_transform = Compose(
    [
        Resize(image_processor.size["shortest_edge"]),
        CenterCrop(image_processor.size["shortest_edge"]),
        ToTensor(),
        normalize,
    ]
)


def train_transforms(examples):
    examples["pixel_values"] = [train_transform(image.convert("RGB")) for image in examples["image"]]

    return examples


In [ ]:
def transform_single_example(example):
    example["pixel_values"] = train_transform(example["image"])
    return example


processed_dataset = DatasetDict(
    {
        "train": dataset["train"].map(transform_single_example),
        "validation": dataset["validation"].map(transform_single_example),
    }
)
print(processed_dataset["train"][0].keys())


In [ ]:
processed_dataset


In [ ]:
processed_dataset["validation"][0].keys()


In [ ]:
def collate_fn(examples):
    pixel_values = torch.stack(
        [
            torch.tensor(example["pixel_values"])
            if not isinstance(example["pixel_values"], torch.Tensor)
            else example["pixel_values"]
            for example in examples
        ]
    )
    labels = torch.tensor([example["label"] for example in examples])
    return {"pixel_values": pixel_values, "labels": labels}


train_dataloader = DataLoader(
    processed_dataset["train"],
    collate_fn=collate_fn,
    batch_size=32,
    shuffle=True,
)
val_dataloader = DataLoader(
    processed_dataset["validation"],
    collate_fn=collate_fn,
    batch_size=32,
    shuffle=False,
)


In [ ]:
batch = next(iter(val_dataloader))
for k, v in batch.items():
    print(k, v.shape)


In [ ]:
model = AutoModelForImageClassification.from_pretrained(
    "facebook/convnext-small-224",
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)


In [ ]:
def train_model(
    model,
    train_dataloader,
    val_dataloader,
    device,
    epochs=20,
    lr=1e-5,
    save_path="best_model.pt",
):
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        patience=2,
        factor=0.5,
    )
    model.classifier = nn.Sequential(nn.Dropout(p=0.2), model.classifier)
    history = {
        "train_acc": [],
        "val_acc": [],
        "train_loss": [],
        "val_loss": [],
    }

    best_val_acc = 0.0
    no_improvement_counter = 0

    for epoch in range(epochs):
        print(f"Epoch {epoch + 1}/{epochs}")
        model.train()
        total = 0
        correct = 0
        running_loss = 0

        for idx, batch in enumerate(tqdm(train_dataloader)):
            batch = {k: v.to(device) for k, v in batch.items()}

            optimizer.zero_grad()
            outputs = model(pixel_values=batch["pixel_values"], labels=batch["labels"])
            loss, logits = outputs.loss, outputs.logits
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            total += batch["labels"].size(0)
            predicted = logits.argmax(dim=1)
            correct += (predicted == batch["labels"]).sum().item()

            if idx % 100 == 0:
                print(f"[Batch {idx}] Train Loss: {loss.item():.4f} | Accuracy: {correct / total:.4f}")

        train_acc = correct / total
        train_loss = running_loss / len(train_dataloader)
        val_acc, val_loss = evaluate(model, val_dataloader, device)
        scheduler.step(val_acc)

        print(f" Epoch {epoch + 1} done — Train acc: {train_acc:.4f} | Val acc: {val_acc:.4f}")

        current_lr = optimizer.param_groups[0]["lr"]
        print(f"Current learning rate: {current_lr:.6f}")
        history["train_acc"].append(train_acc)
        history["train_loss"].append(train_loss)
        history["val_acc"].append(val_acc)
        history["val_loss"].append(val_loss)
        history.setdefault("lr", []).append(current_lr)

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)
            print(f"Best model saved at epoch {epoch} — Val acc improved to {val_acc:.4f}")
            no_improvement_counter = 0
        else:
            no_improvement_counter += 1
            print(f"No improvement. Counter: {no_improvement_counter}")

    return history


def evaluate(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0
    running_loss = 0.0
    with torch.no_grad():
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(pixel_values=batch["pixel_values"], labels=batch["labels"])
            loss, logits = outputs.loss, outputs.logits
            predicted = logits.argmax(dim=1)
            correct += (predicted == batch["labels"]).sum().item()
            total += batch["labels"].size(0)
            running_loss += loss.item()
    return correct / total, running_loss / len(dataloader)


def plot_confusion_matrix(model, dataloader, device, class_names):
    y_true, y_pred = [], []
    model.eval()
    with torch.no_grad():
        for batch in dataloader:
            inputs = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)
            outputs = model(pixel_values=inputs)
            preds = outputs.logits.argmax(dim=1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(xticks_rotation=45)
    plt.title("Confusion Matrix")
    plt.grid(False)
    plt.tight_layout()
    plt.show()


def plot_training_curves(history):
    epochs = range(1, len(history["train_acc"]) + 1)

    plt.figure(figsize=(14, 5))

    # Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_acc"], label="Train Accuracy")
    plt.plot(epochs, history["val_acc"], label="Validation Accuracy")
    plt.title("Accuracy over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True)

    # Loss
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["train_loss"], label="Train Loss")
    plt.plot(epochs, history["val_loss"], label="Validation Loss")
    plt.title("Loss over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)

    # Learning Rate
    if "lr" in history:
        plt.figure()
        plt.plot(
            range(1, len(history["lr"]) + 1),
            history["lr"],
            label="Learning Rate",
        )
        plt.title("Learning Rate over Epochs")
        plt.xlabel("Epoch")
        plt.ylabel("LR")
        plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

history = train_model(
    model=model,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    device=device,
    epochs=20,
    lr=1e-5,
    save_path="best_convnext6.pt",
)


plot_training_curves(history)
plot_confusion_matrix(model, val_dataloader, device, class_names=labels)
